# Символьное дифференцирование

Символьное дифференцирование это инструмент для автоматического вывода формул производных, который открывает возможности для анализа сложных функций, оптимизации процессов и работы с уравнениями. Мы уже на многих занятиях сталкивались с этой темой - давайте попробуем реализовать собственное!

## Выражение

Создадим основной класс `Expr`, от которого будут наследоваться различные типы выражений, такие как константы, переменные, суммы, произведения и другие. Класс должен содержать методы:
* `__call__`, который будет вычислять значение выражения, используя переданный ему контекст (словарь, связывающий имена переменных с их значениями).
* `d`, принимающий имя переменной, по которой требуется вычислить производную, и возвращающий выражение, представляющее производную по этой переменной.

Эти методы нужно будет переопределить в каждом из подклассов для корректного выполнения операций.

In [14]:
class Expr:
    def __call__(self, **context):
        raise NotImplementedError

    def d(self, wrt):
        raise NotImplementedError

    # Перегрузка операторов
    def __neg__(self):
        return Neg(self)

    def __pos__(self):
        return self

    def __add__(self, other):
        if isinstance(other, Expr):
            return Sum(self, other)
        return Sum(self, Const(other))

    def __sub__(self, other):
        if isinstance(other, Expr):
            return Sum(self, Neg(other))
        return Sum(self, Neg(Const(other)))

    def __mul__(self, other):
        if isinstance(other, Expr):
            return Product(self, other)
        return Product(self, Const(other))

    def __truediv__(self, other):
        if isinstance(other, Expr):
            return Fraction(self, other)
        return Fraction(self, Const(other))

    def __repr__(self):
        return f"<Expr>"

Создайте классы для двух видов выражений: `Const`, представляющий константу, и` Var`, представляющий переменную. Чтобы упростить использование, вместо обращения к конструкторам этих классов, будем использовать их однобуквенные сокращённые обозначения.

**Пример использования:**
```python
V = Var
C = Const

C(5)()
5
C(5).d(V("x"))()
0
V("x")(x=5)
5
V("x").d(V("y"))(x=5)
0
V("x").d(V("x"))(x=5)
1
```

In [15]:
class Const(Expr):
    def __init__(self, value):
        self.value = value

    def __call__(self, **context):
        return self.value

    def d(self, wrt):
        return Const(0)

    def __repr__(self):
        return f"Const({self.value})"

In [16]:
class Var(Expr):
    def __init__(self, name):
        self.name = name

    def __call__(self, **context):
        return context[self.name]

    def d(self, wrt):
        return Const(1) if self.name == wrt.name else Const(0)

    def __repr__(self):
        return f"Var({self.name})"

In [29]:
V = Var
C = Const

print(C(5)())
print(C(5).d(V("x"))())
print(V("x")(x=5))
print(V("x").d(V("y"))(x=5))
print(V("x").d(V("x"))(x=5))

5
0
5
0
1


## Бинарные операции

Создайте классы для бинарных операций: `Sum`, `Product` и `Fraction`. Поскольку бинарные операции определяются двумя операндами, их конструктор будет одинаковым для всех этих классов. Поэтому его можно вынести в отдельный базовый класс, чтобы избежать дублирования кода.

In [17]:
class BinOp(Expr):
    def __init__(self, expr1, expr2):
        self.expr1, self.expr2 = expr1, expr2

Реализуйте `Sum` для суммирования, `Product` для умножения и `Fraction` для деления.

**Пример использования:**

```python
Sum(V("x"), Fraction(V("x"), V("y")))(x=5, y=2.5)
7.0
Fraction(Sum(C(5), V("y")), Product(V("x"), V("y")))(x=1, y=2)
3.5
Fraction(Sum(C(5), V("y")), Product(V("x"), V("y"))).d(V("x"))(x=1, y=2)
-3.5
Fraction(Sum(C(5), V("y")), Product(V("x"), V("y"))).d(V("y"))(x=1, y=2)
-1.25
```

In [22]:
class Sum(BinOp):
    def __call__(self, **context):
        return self.expr1(**context) + self.expr2(**context)

    def d(self, wrt):
        return Sum(self.expr1.d(wrt), self.expr2.d(wrt))

    def __repr__(self):
        return f"Sum({repr(self.expr1)}, {repr(self.expr2)})"

In [23]:
x = Var("x")
y = Var("y")

# Пример 1: Sum(V("x"), Fraction(V("x"), V("y")))
expr1 = Sum(x, Fraction(x, y))
result1 = expr1(x=5, y=2.5)
print(result1)

# Пример 2: Fraction(Sum(C(5), V("y")), Product(V("x"), V("y")))
expr2 = Fraction(Sum(Const(5), y), Product(x, y))
result2 = expr2(x=1, y=2)
print(result2)

# Пример 3: Производная по x от Fraction(Sum(C(5), V("y")), Product(V("x"), V("y")))
derivative_x = expr2.d(x)
result3 = derivative_x(x=1, y=2)
print(result3)

# Пример 4: Производная по y от Fraction(Sum(C(5), V("y")), Product(V("x"), V("y")))
derivative_y = expr2.d(y)
result4 = derivative_y(x=1, y=2)
print(result4)

7.0
3.5
3.5
2.25


## Перегрузка операторов

Добавьте перегрузку операторов в базовых класс `Expr`. Обратите что в классах мы можем тоже заменить на использование операторов.
```python  
-e         e.__neg__()
+e         e.__pos__()
e1 + e2    e1.__add__(e2)
e1 - e2    e1.__sub__(e2)
e1 * e2    e1.__mul__(e2)
e1 / e2    e1.__truediv__(e2)
```

**Пример использования:**

```python
(V("x") * V("x") / V("y"))(x=5, y=2.5)
10.0
```

In [24]:
x = Var("x")
y = Var("y")

# Выражение (V("x") * V("x") / V("y"))
f = (x * x) / y

result = f(x=5, y=2.5)
print(result)  # 10.0

10.0


## Метод Ньютона-Рафсона

Напишите функцию `newton_raphson`, которая принимает дифференцируемую функцию  $f$  от переменной  $x$ , начальное приближение  $x_0$ , и положительное число  $\epsilon$ , задающее точность вычислений. Функция должна возвращать значение  $x$ , при котором  $f(x)$  становится равным нулю. Метод Ньютона-Рафсона выполняет итеративный поиск корня функции  $f(x)$ , начиная с начального значения  $x_0$ , и использует правило  
$$x_{n+1} = x_n - \frac{f(x_n)}{f{\prime}(x_n)}$$  
для обновления  $x$  на каждом шаге. Итерации продолжаются до тех пор, пока условие остановки  $|x_{n+1} - x_n| \leq \epsilon$  не будет выполнено.

**Пример использования:**

```python
x = Var("x")
f = Const(-5) * x * x * x * x * x + Const(3) * x + Const(2)
zero = newton_raphson(f, 0.5, eps=1e-4)
zero, f(x=zero)
(1.000000000001132, -2.490496697760136e-11)
```

In [25]:
def newton_raphson(f, x0, eps=1e-4):
    x = x0
    while True:
        f_value = f(x=x)
        f_derivative = f.d(Var("x"))(x=x)

        if f_derivative == 0:
            raise ValueError("Производная равна нулю, деление на ноль.")

        x_new = x - f_value / f_derivative
        if abs(x_new - x) <= eps:
            return x_new
        x = x_new

In [26]:
x = Var("x")
f = Const(-5) * x * x * x * x * x + Const(3) * x + Const(2)

zero = newton_raphson(f, 0.5, eps=1e-4)
print(zero)
print(f(x=zero))

1.0000000000000653
-1.4384049507043528e-12
